### Dependnecy Setup

In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import pyomo.environ as pyo

### Load Data

In [2]:
# Choose dataset B or C
DATA = "B"

In [3]:
df_gen_params = pd.read_csv(f'Data/Data{DATA}_generatorparams.csv', sep=';')
df_load = pd.read_csv(f'Data/Data{DATA}_hourlyload.csv', sep=';')
df_nondisp = pd.read_csv(f'Data/Data{DATA}_nondispgeneration.csv', sep=';')

# If DataC is in MW, convert to kW
# I BELIEVE!
if DATA == "C":
    df_gen_params[['pmin', 'pmax']] *= 1000  # MW → kW
    df_load['load'] *= 1000  # MWh → kWh

### Pre-process Non-dispatchable Data (Calculate Expected Values)

To start off, I calculated the expected values for wind and solar using the data from each time window.<br>
This is something that can be changed in the future.

In [4]:
wind_expected = {}
solar_expected = {}

# Helper function to parse ranges like "0-5"
def parse_range(r):
    start, end = map(int, r.split('-'))
    return range(start, end + 1)

# Group by 'hour' column to sum the expected values for that block
grouped = df_nondisp.groupby('hour')

for hour_range_str, group in grouped:
    # Calculate the expected value for this group
    # E = sum(generation * probability)

    # Wind expected value 
    wind_exp = (group['generation wind'] * group['probability wind']).sum()

    # Solar expected value
    solar_exp = (group['generation solar'] * group['probability solar']).sum()

    # Assign this value to every hour in the range
    for hour in parse_range(hour_range_str):
        wind_expected[hour] = wind_exp
        solar_expected[hour] = solar_exp

# Create dataframe for hourly data
hourly_data = pd.DataFrame({
    'load': df_load.set_index('hour')['load'].astype(float),
    'wind_exp': pd.Series(wind_expected).sort_index(),
    'solar_exp': pd.Series(solar_expected).sort_index()
})

print(hourly_data)

     load  wind_exp  solar_exp
0   496.0      62.0        0.0
1   476.0      62.0        0.0
2   468.0      62.0        0.0
3   472.0      62.0        0.0
4   486.0      62.0        0.0
5   520.0      62.0        0.0
6   584.0      62.0       55.0
7   622.0      62.0       55.0
8   655.0      62.0       55.0
9   671.0      62.0       55.0
10  689.0      62.0       55.0
11  701.0      62.0      104.0
12  693.0      62.0      104.0
13  686.0      62.0      104.0
14  647.0      62.0      104.0
15  626.0      62.0       55.0
16  613.0      62.0       55.0
17  616.0      62.0       55.0
18  645.0      62.0       55.0
19  644.0      62.0       55.0
20  605.0      62.0        0.0
21  577.0      62.0        0.0
22  551.0      62.0        0.0
23  520.0      62.0        0.0


### Model Initialization

In [5]:
model = pyo.ConcreteModel(name="Unit Commitment Stage 1")

# Sets
model.S = pyo.Set(initialize=range(20)) # Sample over 20 scenarios

# Generators [1, 2, 3]
model.G = pyo.Set(initialize=df_gen_params['generator'].tolist())

# Time steps [0, 1, ..., 23]
model.T = pyo.Set(initialize=hourly_data.index.tolist())

### Parameters

In [6]:
# Parameters for Generators 
model.a1 = pyo.Param(model.G, initialize=df_gen_params.set_index('generator')['a1'].to_dict())
model.a2 = pyo.Param(model.G, initialize=df_gen_params.set_index('generator')['a2'].to_dict())
model.p_min = pyo.Param(model.G, initialize=df_gen_params.set_index('generator')['pmin'].to_dict())
model.p_max = pyo.Param(model.G, initialize=df_gen_params.set_index('generator')['pmax'].to_dict())

# Parameters for Hourly Data 
model.hourly_load = pyo.Param(model.T, initialize=hourly_data['load'].to_dict())
model.wind_exp = pyo.Param(model.T, initialize=hourly_data['wind_exp'].to_dict())
model.solar_exp = pyo.Param(model.T, initialize=hourly_data['solar_exp'].to_dict())

### Decision Variables

In [7]:
# u[g, t]: Binary status of dispatchable generator g at time t (1 if on, 0 if off)
model.u = pyo.Var(model.G, model.T, domain=pyo.Binary)

# p[g, t]: Power output of dispatchable generator g at time t (non-negative) in scenario S
model.p = pyo.Var(model.G, model.T, model.S, domain=pyo.NonNegativeReals)

# u_wind[t], u_solar[t]: Binary status of non-dispatchable generators at time t
model.u_wind = pyo.Var(model.T, domain=pyo.Binary)
model.u_solar = pyo.Var(model.T, domain=pyo.Binary)

## Slack Variables

In [8]:
# surplus[t], deficit[t]: Surplus and deficit variables at time t 
# to ensure feasibility if production does not equal demand
model.surplus = pyo.Var(model.T, model.S, domain=pyo.NonNegativeReals)
model.deficit = pyo.Var(model.T, model.S, domain=pyo.NonNegativeReals)

### Objective Function
Minimize the cost of power generators averaged over the scenarios S

In [9]:
def objective_rule(m):
    total_expected_cost = 0
    for s in m.S:
        generation_cost = sum(m.a2[g] * m.p[g,t,s]**2 + m.a1[g] * m.p[g,t,s] for g in m.G for t in m.T)
        penalty_cost = sum(1000 * (m.surplus[t,s] + m.deficit[t,s]) for t in m.T)
        total_expected_cost += (generation_cost + penalty_cost)
    return total_expected_cost / len(m.S)
model.obj = pyo.Objective(rule=objective_rule, sense=pyo.minimize)

### Constraints

In [10]:
# Minimum Generation
def p_min_rule(m, g, t, s):
    return m.p[g, t, s] >= m.u[g, t] * m.p_min[g]
model.C_p_min = pyo.Constraint(model.G, model.T, model.S, rule=p_min_rule)

# Maximum Generation
def p_max_rule(m, g, t, s):
    return m.p[g, t, s] <= m.u[g, t] * m.p_max[g]
model.C_p_max = pyo.Constraint(model.G, model.T, model.S, rule=p_max_rule)

# Power Balance
def p_balance_rule(m, t, s):
    dispatchable_gen = sum(m.p[g, t, s] for g in m.G)
    nondispatchable_gen = (m.u_wind[t] * m.wind_exp[t]) + (m.u_solar[t] * m.solar_exp[t])

    return dispatchable_gen + nondispatchable_gen + m.surplus[t, s] - m.deficit[t, s] == m.hourly_load[t]
model.C_p_balance = pyo.Constraint(model.T, model.S, rule=p_balance_rule)

### Solving with NEOS

In [11]:
# Setup Solver Manager
solver_manager = pyo.SolverManagerFactory('neos')

# Select Solver
solver_name = 'cplex' 

# Your email address (required for NEOS)
os.environ['NEOS_EMAIL'] = 'jonas.streuff@tum.de'

print(f"Sending model to NEOS using {solver_name}...")

# Send to NEOS
results = solver_manager.solve(model, opt=solver_name)
print(f"Solver Status: {results.solver.status}")
print(f"Termination Condition: {results.solver.termination_condition}")

# If successful, we extract the results
if results.solver.termination_condition == pyo.TerminationCondition.optimal:
    print("Optimal solution found.")
else:
    print("Warning: Optimal solution not found.")


Sending model to NEOS using cplex...
Solver Status: ok
Termination Condition: optimal
Optimal solution found.


### Print Results

In [13]:
def get_timeseries(variable, generator_idx=None, is_binary=False, has_scenarios=False):
    data = []
    for t in sorted(model.T):
        if has_scenarios:
            # Average across all scenarios
            if generator_idx is not None:
                val = sum(variable[generator_idx, t, s].value or 0 for s in model.S) / len(model.S)
            else:
                val = sum(variable[t, s].value or 0 for s in model.S) / len(model.S)
        else:
            if generator_idx is not None:
                val = variable[generator_idx, t].value
            else:
                val = variable[t].value
        
        if val is None:
            val = 0.0
        if is_binary:
            val = round(val)
        data.append(val)
    return data

load_data = [model.hourly_load[t] for t in sorted(model.T)]
wind_exp = [model.wind_exp[t] for t in sorted(model.T)]
solar_exp = [model.solar_exp[t] for t in sorted(model.T)]

u_wind = get_timeseries(model.u_wind, is_binary=True)
u_solar = get_timeseries(model.u_solar, is_binary=True)

gen_ids = sorted(model.G)
p_gen_data = {}
u_gen_data = {}

for g in gen_ids:
    p_gen_data[g] = get_timeseries(model.p, generator_idx=g, has_scenarios=True)
    u_gen_data[g] = get_timeseries(model.u, generator_idx=g, is_binary=True)

# 3. Print the Formatted Table
print("\n" + "="*90)
print(f"OPTIMIZATION RESULTS - Total Cost: {pyo.value(model.obj):,.2f} €")
print("="*90)

# Build Header
# Fixed Columns
header = f"{'Hr':<3} | {'Load':<6} | {'Wind':<5} {'U':<2} | {'Solar':<5} {'U':<2} | "
# Dynamic Columns (one pair per generator)
for g in gen_ids:
    header += f"G{g} (P)   G{g}(U) | "
print(header)
print("-" * len(header))

# Print Rows
for t in range(24):
    # Base Data (Load + Renewables)
    # We cast to int/float for clean formatting
    row = (f"{t:<3} | {load_data[t]:<6.0f} | "
           f"{wind_exp[t]:<5.0f} {int(u_wind[t]):<2} | "
           f"{solar_exp[t]:<5.0f} {int(u_solar[t]):<2} | ")
    
    # Generator Data
    for g in gen_ids:
        p_val = p_gen_data[g][t]
        u_val = int(u_gen_data[g][t])
        
        # Visual cleanup: if generator is OFF, show Power as 0.0
        if u_val == 0:
            p_val = 0.0
            
        row += f"{p_val:<8.1f} {u_val:<5} | "
    
    print(row)

print("-" * len(header))


OPTIMIZATION RESULTS - Total Cost: 371,161.43 €
Hr  | Load   | Wind  U  | Solar U  | G1 (P)   G1(U) | G2 (P)   G2(U) | G3 (P)   G3(U) | 
----------------------------------------------------------------------------------------
0   | 496    | 62    1  | 0     0  | 200.0    1     | 180.0    1     | 54.0     1     | 
1   | 476    | 62    1  | 0     0  | 252.4    1     | 0.0      0     | 161.6    1     | 
2   | 468    | 62    1  | 0     0  | 247.6    1     | 0.0      0     | 158.4    1     | 
3   | 472    | 62    1  | 0     0  | 250.0    1     | 0.0      0     | 160.0    1     | 
4   | 486    | 62    1  | 0     0  | 200.0    1     | 180.0    1     | 44.0     1     | 
5   | 520    | 62    1  | 0     0  | 200.0    1     | 180.0    1     | 78.0     1     | 
6   | 584    | 62    1  | 55    1  | 200.0    1     | 180.0    1     | 87.0     1     | 
7   | 622    | 62    1  | 55    1  | 200.0    1     | 180.0    1     | 125.0    1     | 
8   | 655    | 62    1  | 55    1  | 218.8    1     | 180.0  

### Plot Results

In [ ]:
hours = sorted(model.T)

# Dispatchable Generation
wind_gen = []
solar_gen = []
dispatchable_gen = {g: [] for g in gen_ids}

for t in hours:
    wind_gen.append(u_wind[t] * wind_exp[t])
    solar_gen.append(u_solar[t] * solar_exp[t])

    for g in gen_ids:
        dispatchable_gen[g].append(p_gen_data[g][t])


plt.figure(figsize=(12,6))

plt.stackplot(hours, wind_gen, solar_gen, dispatchable_gen[1], dispatchable_gen[2], dispatchable_gen[3],
              labels=['Wind Generation', 'Solar Generation', 'Generator 1', 'Generator 2', 'Generator 3'],
              colors=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'])

plt.plot(hours, load_data, color='black', linewidth=3, label='Load Profile', linestyle='--')

plt.title("Unit Commitment - Generation Schedule", fontsize=14)
plt.xlabel("Hour of Day", fontsize=12)
plt.ylabel("Power Generation (kW)", fontsize=12)
plt.xlim(0, 23)
plt.xticks(hours)
plt.grid(True, alpha=0.3)
plt.legend(loc='upper left', bbox_to_anchor=(1,1))
plt.tight_layout()
plt.show()